<a href="https://colab.research.google.com/github/KyeongHoSeong/rl_studios/blob/main/002_01_Softmax_One_hot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Reference: [PyTorch로 시작하는 딥 러닝 입문-5.2]-(https://wikidocs.net/59427) 

가설 : H(X)=sigmoid(WX+B)
<img src="https://wikidocs.net/images/page/59427/%EA%B0%80%EC%84%A4.PNG">


W: 3x4, X: 4x1 + B: 3x1 = Y: 3x1
X: 5x4  W: 4x3   B: 5x3 = Y: 5X3

In [82]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

1.Datasoftmax test

In [96]:
z = torch.FloatTensor([1,2,3])
hypothesis = F.softmax(z, dim=0)

print(hypothesis)
hypothesis.sum()

tensor([0.0900, 0.2447, 0.6652])


tensor(1.)

In [84]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

2. cost

In [101]:
z = torch.rand(3, 5, requires_grad=True)
hypothesis = F.softmax(z, dim=1)
print(hypothesis)

y = torch.randint(5, (3,)).long()
print(y)

tensor([[0.2009, 0.2406, 0.2124, 0.1176, 0.2286],
        [0.1568, 0.2733, 0.2362, 0.1922, 0.1415],
        [0.1332, 0.1255, 0.1899, 0.3056, 0.2458]], grad_fn=<SoftmaxBackward>)
tensor([3, 3, 2])


3. one hot encoding

In [105]:
print(hypothesis)
# 모든 원소가 0의 값을 가진 3 × 5 텐서 생성
y_one_hot = torch.zeros_like(hypothesis) 

print( y.unsqueeze(1))
#  y.unsqueeze(1)--> tensor(3 X 1)

# dim=1에 대해서 수행, y.unsqueeze(1)이 알려주는 위치에 1을 넗어라
y_one_hot.scatter_(1, y.unsqueeze(1), 1)
print(y_one_hot)

tensor([[0.2009, 0.2406, 0.2124, 0.1176, 0.2286],
        [0.1568, 0.2733, 0.2362, 0.1922, 0.1415],
        [0.1332, 0.1255, 0.1899, 0.3056, 0.2458]], grad_fn=<SoftmaxBackward>)
tensor([[3],
        [3],
        [2]])
tensor([[0., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0.]])


In [106]:
cost = (y_one_hot * -torch.log(hypothesis)).sum(dim=1).mean()
print(cost)

tensor(1.8170, grad_fn=<MeanBackward0>)


# 2. 파이토치로 소프트맥스의 비용 함수 구현하기 (하이-레벨)


In [109]:
# 1. F.softmax() + torch.log() = F.log_softmax()
# Low level
torch.log(F.softmax(z, dim=1))


tensor([[-1.6050, -1.4248, -1.5494, -2.1403, -1.4760],
        [-1.8526, -1.2972, -1.4430, -1.6493, -1.9557],
        [-2.0161, -2.0752, -1.6614, -1.1854, -1.4033]], grad_fn=<LogBackward>)

In [110]:
# High level
F.log_softmax(z, dim=1)

tensor([[-1.6050, -1.4248, -1.5494, -2.1403, -1.4760],
        [-1.8526, -1.2972, -1.4430, -1.6493, -1.9557],
        [-2.0161, -2.0752, -1.6614, -1.1854, -1.4033]],
       grad_fn=<LogSoftmaxBackward>)

# 2. F.log_softmax() + F.nll_loss() = F.cross_entropy()

In [111]:
# Low level
# 첫번째 수식
(y_one_hot * -torch.log(F.softmax(z, dim=1))).sum(dim=1).mean()

tensor(1.8170, grad_fn=<MeanBackward0>)

In [112]:
# 두번째 수식
(y_one_hot * - F.log_softmax(z, dim=1)).sum(dim=1).mean()

tensor(1.8170, grad_fn=<MeanBackward0>)

In [113]:
# High level
# 세번째 수식
F.nll_loss(F.log_softmax(z, dim=1), y)

tensor(1.8170, grad_fn=<NllLossBackward>)

In [114]:
# 네번째 수식
F.cross_entropy(z, y)

tensor(1.8170, grad_fn=<NllLossBackward>)

04. 소프트맥스 회귀 구현하기

In [115]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [116]:
x_train = [[1, 2, 1, 1],
           [2, 1, 3, 2],
           [3, 1, 3, 4],
           [4, 1, 5, 5],
           [1, 7, 5, 5],
           [1, 2, 5, 6],
           [1, 6, 6, 6],
           [1, 7, 7, 7]]
y_train = [2, 2, 2, 1, 1, 1, 0, 0]
x_train = torch.FloatTensor(x_train)
y_train = torch.LongTensor(y_train)

In [117]:
print(x_train.shape)
print(y_train.shape)

torch.Size([8, 4])
torch.Size([8])


In [118]:
y_one_hot = torch.zeros(8, 3)
y_one_hot.scatter_(1, y_train.unsqueeze(1), 1)
print(y_one_hot.shape)

torch.Size([8, 3])


In [119]:
# 모델 초기화
W = torch.zeros((4, 3), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# optimizer 설정
optimizer = optim.SGD([W, b], lr=0.1)

In [120]:
nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # 가설
    hypothesis = F.softmax(x_train.matmul(W) + b, dim=1) 

    # 비용 함수
    cost = (y_one_hot * -torch.log(hypothesis)).sum(dim=1).mean()

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 100번마다 로그 출력
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/1000 Cost: 1.098612
Epoch  100/1000 Cost: 0.761050
Epoch  200/1000 Cost: 0.689991
Epoch  300/1000 Cost: 0.643229
Epoch  400/1000 Cost: 0.604117
Epoch  500/1000 Cost: 0.568255
Epoch  600/1000 Cost: 0.533922
Epoch  700/1000 Cost: 0.500291
Epoch  800/1000 Cost: 0.466908
Epoch  900/1000 Cost: 0.433507
Epoch 1000/1000 Cost: 0.399962


2. 소프트맥스 회귀 구현하기(하이-레벨)

In [121]:
# 모델 초기화
W = torch.zeros((4, 3), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# optimizer 설정
optimizer = optim.SGD([W, b], lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # Cost 계산
    z = x_train.matmul(W) + b
    cost = F.cross_entropy(z, y_train)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 100번마다 로그 출력
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/1000 Cost: 1.098612
Epoch  100/1000 Cost: 0.761050
Epoch  200/1000 Cost: 0.689991
Epoch  300/1000 Cost: 0.643229
Epoch  400/1000 Cost: 0.604117
Epoch  500/1000 Cost: 0.568255
Epoch  600/1000 Cost: 0.533922
Epoch  700/1000 Cost: 0.500291
Epoch  800/1000 Cost: 0.466908
Epoch  900/1000 Cost: 0.433507
Epoch 1000/1000 Cost: 0.399962


3. 소프트맥스 회귀 nn.Module로 구현하기

In [122]:
# 모델을 선언 및 초기화. 4개의 특성을 가지고 3개의 클래스로 분류. input_dim=4, output_dim=3.
model = nn.Linear(4, 3)

In [123]:
# optimizer 설정
optimizer = optim.SGD(model.parameters(), lr=0.1)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    prediction = model(x_train)

    # cost 계산
    cost = F.cross_entropy(prediction, y_train)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 20번마다 로그 출력
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(
            epoch, nb_epochs, cost.item()
        ))

Epoch    0/1000 Cost: 1.616785
Epoch  100/1000 Cost: 0.658891
Epoch  200/1000 Cost: 0.573443
Epoch  300/1000 Cost: 0.518151
Epoch  400/1000 Cost: 0.473265
Epoch  500/1000 Cost: 0.433516
Epoch  600/1000 Cost: 0.396563
Epoch  700/1000 Cost: 0.360914
Epoch  800/1000 Cost: 0.325392
Epoch  900/1000 Cost: 0.289178
Epoch 1000/1000 Cost: 0.254148
